# Information Coefficient & t-stats lab

The backtest asks *did the trades make money?* IC asks the sharper
question: *does a higher score predict a better outcome among that day's
candidates?* -- the Grinold/Kahn signal-quality measure. A strategy can
earn positive avg R while its score ranks nothing (all edge in the exit
engine), or rank well while costs eat the money. IC tells you **where**
to work.

Rules of thumb (daily equities): |mean IC| ~0.02 weak, ~0.05 useful,
>0.10 suspicious (check leakage). t-stat >= 2 on the daily IC series =
real ranking skill. Overlapping h-day horizons inflate t by ~sqrt(h) --
trust the **overlap-adjusted** column.

In [ ]:
from pathlib import Path
import subprocess, sys, json
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd() if (Path.cwd() / 'config.json').exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
from rhengine import viz
viz.use_style()
CFG = json.load(open(ROOT / 'config.json'))
RESULTS = ROOT / 'tests' / 'results'; RESULTS.mkdir(parents=True, exist_ok=True)

# Full-market event-loop passes are slow; cap the universe for a live
# notebook run and raise it (or set 0) when you want the real number.
MAX_TICKERS  = 150
LOOKBACK     = 400
COST_BPS     = 5
REFRESH      = False   # True -> recompute cached CSVs even if present
print('root:', ROOT, '| data_source:', CFG.get('data_source'))

In [ ]:
STRATEGY  = None            # None = auto-regime (all sleeves); or force one
SIGNAL    = 'weighted_score'  # 'weighted_score' | 'score'
HORIZONS  = (1, 5, 10, 20)
MIN_NAMES = 8               # min candidates per day for a valid daily IC

## 1 -- Harvest the scored signal panel

One backtest at the emission floor, each signal tagged with its score and
forward returns (raw and SPY-excess) at every horizon. Cached to CSV.

In [ ]:
from rhengine.ic import harvest_signals, daily_ic, ic_table, ic_summary, score_tier_returns
from rhengine.data_local import load_bars_any
PANEL = RESULTS / f'ic_panel_{STRATEGY or "auto"}.csv'
if REFRESH or not PANEL.exists():
    bars_map, spy_bars = load_bars_any(CFG, lookback_days=LOOKBACK, verbose=False)
    if MAX_TICKERS: bars_map = dict(list(bars_map.items())[:MAX_TICKERS])
    panel = harvest_signals(CFG, bars_map, spy_bars, min_score=2,
                            forced_strategy=STRATEGY, cost_bps=COST_BPS,
                            horizons=HORIZONS)
    panel.to_csv(PANEL, index=False)
else:
    panel = pd.read_csv(PANEL)
print(len(panel), 'signals |', panel['strategy'].nunique(), 'strategies')
panel.head()

## 2 -- IC summary table

Mean IC, its t-stat (raw and overlap-adjusted), IC information ratio,
and %-positive days -- per horizon, against SPY-excess returns. The
`r_multiple` row is the exit-engine outcome (what the backtest scores).

In [ ]:
ic_exc = ic_table(panel, signal=SIGNAL, horizons=HORIZONS, min_names=MIN_NAMES, kind='exc')
if ic_exc.empty:
    print('No horizon had enough cross-sectional days -- raise MAX_TICKERS,',
          'widen the window, or lower MIN_NAMES.')
display(ic_exc.style.format('{:+.4f}') if hasattr(ic_exc, 'style') and not ic_exc.empty else ic_exc)

In [ ]:
# t-stat (adjusted) per horizon -- polarity bars. |t|>=2 is the bar.
if not ic_exc.empty and 't_stat_adj' in ic_exc.columns:
    fig, ax = viz.bars_signed(ic_exc.index.tolist(), ic_exc['t_stat_adj'].tolist(),
            title=f'{STRATEGY or "all sleeves"} -- overlap-adjusted IC t-stat by horizon',
            xlabel='t-stat (|t|>=2 = real skill)', fmt='{:+.2f}')
    ax.axvline(2, color=viz.MUTED, ls=':', lw=1); ax.axvline(-2, color=viz.MUTED, ls=':', lw=1)
    plt.show()
else:
    print('IC table empty -- nothing to chart (see note above).')

## 3 -- Daily IC time series

Is the ranking skill persistent or a couple of lucky days? Faint dots =
daily IC, line = rolling mean, dotted = full-sample mean.

In [ ]:
HZ = 5  # horizon to inspect (days)
s = daily_ic(panel, signal=SIGNAL, target=f'exc_{HZ}', min_names=MIN_NAMES)
if len(s) >= 3:
    st = ic_summary(s, horizon=HZ)
    print({k: round(v,4) for k,v in st.items()})
    viz.ic_timeseries(s, roll=max(5, len(s)//8),
        title=f'{STRATEGY or "all"} daily IC vs {HZ}d SPY-excess (adj t={st["t_stat_adj"]:+.2f})')
    plt.show()
else:
    print('not enough cross-sectional days -- raise MAX_TICKERS or lower MIN_NAMES')

## 4 -- Score-tier monotonicity

The most direct test of whether the score ranks: does 4/4 out-earn 3/4
out-earn 2/4? If the bars are flat or inverted, the extra legs are not
adding ranking information -- feed that straight into the ablation lab.

In [ ]:
tiers = score_tier_returns(panel, horizons=HORIZONS, kind='exc')
display(tiers)
hz_col = (f'exc_{HZ}', 'mean')
if hz_col in tiers.columns:
    ks = [str(k) for k in tiers.index]
    fig, ax = viz.bars_signed([f'score {k}' for k in ks],
        tiers[hz_col].tolist(), title=f'Mean {HZ}d SPY-excess by score tier',
        xlabel=f'{HZ}d excess return', fmt='{:+.4f}')
    plt.show()

---
**Read-off:** positive, significant (adj t>=2) IC with monotone score
tiers = the entry signal genuinely ranks -- refine by *adding* filters
(RRG angle/distance, stage gate) in the walk-forward lab. Flat IC with
positive avg R = the edge is in the exit engine, so refine geometry
(stop/target ATR mults), not entries. Confirm with `nulltest.py`.